# DeepTuneIO — Module 10: Access-Pattern Visualization

Generic visualization of DXT event-level access patterns.

The notebook contains only campaign selection and orchestration. Scientific
visualization logic lives under `Module/Access_Pattern_Visualization/`.


In [ ]:
pwd

In [1]:
from pathlib import Path
import sys
from importlib.util import spec_from_file_location, module_from_spec

MODULE_DIR = Path("./Module").resolve()
MODULE_FILE = MODULE_DIR / "10_access_pattern_visualization.py"

spec = spec_from_file_location("deeptuneio_access_viz10", MODULE_FILE)
module10 = module_from_spec(spec)
sys.modules[spec.name] = module10
spec.loader.exec_module(module10)

print("Module :", MODULE_FILE)
print("Version:", module10.VERSION)


Module : F:\NoteBooks\CORA_DR_UAB\Tesis\Scripts\DeepTuneIO\10_Visualization\Access_Pattern\Module\10_access_pattern_visualization.py
Version: 1.4.1


In [2]:
PROJECT_ROOT = Path(r"D:\DL_Experiments\CORA_DR_UAB").resolve()
ARTICLE_ID = "Tesis"

EVENT = "Tesis"

CAMPAIGNS = {
    "DeepGalaxy": {
        "DG_bw512_f3c5x64": {
            "scenarios": ["lustre_1ost", "lustre_2ost", "lustre_4ost", "nfs_SnGPU_e1"],
            "file_format": ".hdf5",
        },
    },
    "DLIOv1": {
        "HDF5_64bs": {"scenarios": ["lustre_ost"], "file_format": ".h5"},
        "NPZ_64bs": {"scenarios": ["lustre_ost"], "file_format": ".npz"},
        "TFRecord_256kts_64bs": {"scenarios": ["lustre_ost"], "file_format": ".tfrecords"},
        "TFRecord_1mts_64bs": {"scenarios": ["lustre_ost"], "file_format": ".tfrecords"},
    },
}

APPLICATION_POS = 1
DATASET_POS = 1
SCENARIO_POS = 4

APPLICATION = list(CAMPAIGNS)[APPLICATION_POS - 1]
DATASET_GROUP = list(CAMPAIGNS[APPLICATION])[DATASET_POS - 1]
SCENARIO = CAMPAIGNS[APPLICATION][DATASET_GROUP]["scenarios"][SCENARIO_POS - 1]
FILE_FORMAT = CAMPAIGNS[APPLICATION][DATASET_GROUP]["file_format"]

CAMPAIGN_ROOT = (
    PROJECT_ROOT / ARTICLE_ID / "Data"
    / APPLICATION / DATASET_GROUP / SCENARIO
)

print(APPLICATION, DATASET_GROUP, SCENARIO, FILE_FORMAT)
print("Campaign:", CAMPAIGN_ROOT)


DeepGalaxy DG_bw512_f3c5x64 nfs_SnGPU_e1 .hdf5
Campaign: D:\DL_Experiments\CORA_DR_UAB\Tesis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1


## Select the execution dynamically from the experiment matrix

The selector is now **matrix-first**: `experiment_matrix_*.csv` is the source of
truth for configuration metadata, while DXT is used only to confirm that a JobID
has detailed events available.

Normalized modes:
- DeepGalaxy `shared` → `shared`
- DeepGalaxy `shared_reload_shuffle` → `shared_reload_shuffle`
- DLIO `shared` → `shared`
- DLIO `multi` → `file_per_process`


In [3]:
JOB_TABLE = module10.build_job_selection_table(
    CAMPAIGN_ROOT,
    application=APPLICATION,
    scenario=SCENARIO,
    file_format=FILE_FORMAT,
)

DISPLAY_COLUMNS = [
    "job_id",
    "nodes",
    "process_io",
    "replicate",
    "dxt_available",
    "dxt_events",
]

display(
    JOB_TABLE[
        [c for c in DISPLAY_COLUMNS if c in JOB_TABLE.columns]
    ]
)


,job_id,nodes,process_io,replicate,dxt_available,dxt_events
0,4942237,1,4,1,True,262061
1,5175899,1,4,2,True,262061
2,4948089,1,4,1,True,262061
3,5186375,1,4,2,True,262061
4,5317261,1,4,3,True,262061
5,4998168,2,8,1,True,508104
6,5326016,2,8,2,True,508104
7,5346216,2,8,3,True,508104
8,5124492,2,8,1,True,524189
9,5136376,2,8,2,True,524189


In [4]:
# Optional selector.
TARGET_NODES = None
TARGET_PROCESSES = None
TARGET_MODE = "all"       # shared | shared_reload_shuffle | file_per_process | all
TARGET_FILESYSTEM = "all"
TARGET_OST = None

JOB_CANDIDATES = module10.filter_job_selection_table(
    JOB_TABLE,
    nodes=TARGET_NODES,
    process_io=TARGET_PROCESSES,
    data_loading_mode=TARGET_MODE,
    filesystem=TARGET_FILESYSTEM,
    stripe_count=TARGET_OST,
    dxt_only=True,
)

display(JOB_CANDIDATES[[c for c in DISPLAY_COLUMNS if c in JOB_CANDIDATES.columns]])

if len(JOB_CANDIDATES) == 1:
    JOB_ID = int(JOB_CANDIDATES.iloc[0]["job_id"])
    print("Selected JobID:", JOB_ID)
else:
    JOB_ID = None
    print(f"{len(JOB_CANDIDATES)} DXT candidates found. Refine the filters or choose a JobID from the table.")


,job_id,nodes,process_io,replicate,dxt_available,dxt_events
0,4942237,1,4,1,True,262061
1,5175899,1,4,2,True,262061
2,4948089,1,4,1,True,262061
3,5186375,1,4,2,True,262061
4,5317261,1,4,3,True,262061
5,4998168,2,8,1,True,508104
6,5326016,2,8,2,True,508104
7,5346216,2,8,3,True,508104
8,5124492,2,8,1,True,524189
9,5136376,2,8,2,True,524189


25 DXT candidates found. Refine the filters or choose a JobID from the table.


In [16]:
# You may also override the automatic selection manually after inspecting the table:
#JOB_ID = 7296725  # DLIOv1 HDF5 Shared 1N4p L1
#JOB_ID = 7296914 # DLIOv1 HDF5  Shared 16N64p L16
#JOB_ID = 3538214  # DLIOv1 NPZ multi 1N4p L1
#JOB_ID = 3538779 # DLIOv1 NPZ multi 16N64p L16
#JOB_ID = 3528468  # DLIOv1 TFRecord 256K TS multi 1N4p L1
#JOB_ID = 3529616 # DLIOv1 TFRecord 256K TS multi 16N64p L16
#JOB_ID = 3528998  # DLIOv1 TFRecord 1M TS multi 1N4p L1
#JOB_ID = 3529617 # DLIOv1 TFRecord 1M TS multi 16N64p L16


In [6]:
# You may also override the automatic selection manually after inspecting the table:
#JOB_ID = 12157519  # DeepGalaxy  shared_reload_shuffle 32N128p L1
#JOB_ID = 12155384 # DeepGalaxy Shared 32N128p L1
#JOB_ID = 8879555  # DeepGalaxy shared 1N4p L1
#JOB_ID = 8975491 # DeepGalaxy shared_reload_shuffle 1N4p L1
JOB_ID = 5333326  # DeepGalaxy Shared 16N64p NFS
#JOB_ID = 5402981 # DeepGalaxy Shared 16N64p L2


In [15]:
# ============================================================
# Access-pattern visualization profile
# ============================================================
#
# Select which family of DXT access-pattern figures to generate.
#
# Available options:
#
# "process3d"
#     Process-oriented spatial/temporal analysis.
#     Generates:
#       01_spatial_temporal_3d
#          X = I/O Process
#          Y = Operation Order
#          Z = File Offset
#          Color/size = Request Size
#
#       04_spatial_pattern_by_process
#          X = I/O Process
#          Y = File Offset
#          Color/size = Request Size
#
# "temporal2d"
#     Classical 2D spatial and temporal analysis.
#     Generates:
#       02_spatial_pattern
#          X = Operation Order
#          Y = File Offset
#          Color/size = Request Size
#
#       03_temporal_pattern
#          X = Start Time
#          Y = File Offset
#          Color/size = Request Size
#
# "temporal3d"
#     Real-time process-aware spatio-temporal analysis.
#     Generates:
#       07_spatial_temporal_time_3d
#          X = I/O Process
#          Y = Start Time
#          Z = File Offset
#          Color/size = Request Size
#
#       03_temporal_pattern
#          X = Start Time
#          Y = File Offset
#          Color/size = Request Size
#
# "hist"
#     Spatial access pattern with marginal histograms.
#     Generates:
#       05_spatial_pattern_histograms
#
# "ost"
#     DXT-reported OST association overview.
#     Generates:
#       06_ost_association_overview
#
# "all"
#     Generates all available access-pattern figure families:
#       01, 02, 03, 04, 05, 06 and 07.
#
# Examples:
#     PROFILE = "process3d"
#     PROFILE = "temporal2d"
#     PROFILE = "temporal3d"
#     PROFILE = "hist"
#     PROFILE = "ost"
#     PROFILE = "dxt_timeline"
#     PROFILE = "all"
#
# ============================================================

PROFILE = "all" #"process3d"

In [16]:
# You may also override the automatic selection manually after inspecting the table:
# JOB_ID = 4935078

# all | read | write
OPERATION = "read"

# 3D view configuration
VIEW_ELEV = 15
VIEW_AZIM = -30  # 25


# Optional deterministic visual downsampling for very dense traces.
# Keep None for exact full-trace figures.
MAX_EVENTS = None

print("JobID:", JOB_ID)


JobID: 12155384


In [17]:
args = [
    "--campaign-root", str(CAMPAIGN_ROOT),
    "--application", APPLICATION,
    "--dataset-group", DATASET_GROUP,
    "--scenario", SCENARIO,
    "--file-format", FILE_FORMAT,
    "--event", EVENT,
    "--operation", OPERATION,
    "--profile", PROFILE,

    # 3D view
    "--view-elev", str(VIEW_ELEV),
    "--view-azim", str(VIEW_AZIM),
]

if JOB_ID is not None:
    args += ["--job-id", str(JOB_ID)]

if MAX_EVENTS is not None:
    args += ["--max-events", str(MAX_EVENTS)]

return_code = module10.main(args)
print("Return code:", return_code)


DeepTuneIO — Module 10: Access-Pattern Visualization
Campaign root : D:\DL_Experiments\CORA_DR_UAB\Tesis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1
Application   : DeepGalaxy
Dataset group : DG_bw512_f3c5x64
Scenario      : nfs_SnGPU_e1
File format   : .hdf5
JobID         : 12155384
Operation     : read
Profile       : all
Output        : D:\DL_Experiments\CORA_DR_UAB\Tesis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1\results\Figures\Access_Pattern\job_12155384
Return code: 1


ERROR during access-pattern preparation: JobID 12155384 not found in detailed DXT products. Available JobIDs: [4942237, 4948089, 4998168, 5006144, 5124492, 5136376, 5169046, 5172871, 5173308, 5175899, 5179061, 5181620, 5182632, 5185108, 5186375, 5189998, 5192483, 5317261, 5326016, 5333197, 5333326, 5346216, 5352800, 5353626, 5364632]


## Article 01 — Fig. 10(b,d) spatial model

Version 1.1 adds the missing 2D spatial representation:

- **X:** I/O Process
- **Y:** File Offset
- **Color:** Read Request Size

A 4-process JobID reproduces the Fig. 10(b) model; a 64-process JobID reproduces the Fig. 10(d) model.

In [18]:
OUTPUT_ROOT = (
    CAMPAIGN_ROOT / "results" / "Figures" / "Access_Pattern"
    / (f"job_{JOB_ID}" if JOB_ID is not None else "campaign")
)

print("Outputs:", OUTPUT_ROOT)

if OUTPUT_ROOT.exists():
    for p in sorted(OUTPUT_ROOT.iterdir()):
        print(" -", p.name)


Outputs: D:\DL_Experiments\CORA_DR_UAB\Tesis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1\results\Figures\Access_Pattern\job_12155384


In [14]:
from pathlib import Path
import pandas as pd

analysis_dir = (
    CAMPAIGN_ROOT
    / "results"
    / "Access_Pattern_DXT"
    / "Analysis"
)

traces_dir = (
    CAMPAIGN_ROOT
    / "results"
    / "Access_Pattern_DXT"
    / "Traces"
)

JOB_ID = 12155384

analysis_files = sorted(
    analysis_dir.glob(f"*{JOB_ID}*_dxt_analysis.csv")
)

trace_files = sorted(
    traces_dir.glob(f"*{JOB_ID}*_dxt.csv")
)

print("=" * 80)
print("DXT SOURCE COMPARISON")
print("=" * 80)

print("\nAnalysis files:")
for p in analysis_files:
    df = pd.read_csv(p)
    print(f"  {p.name}")
    print(f"    rows    : {len(df)}")
    print(f"    columns : {len(df.columns)}")

print("\nTrace files:")
for p in trace_files:
    df = pd.read_csv(p)
    print(f"  {p.name}")
    print(f"    rows    : {len(df)}")
    print(f"    columns : {len(df.columns)}")

if analysis_files and trace_files:

    a = pd.read_csv(analysis_files[0])
    t = pd.read_csv(trace_files[0])

    print("\n" + "=" * 80)
    print("STRUCTURAL COMPARISON")
    print("=" * 80)

    print("Analysis rows :", len(a))
    print("Trace rows    :", len(t))

    common = sorted(
        set(a.columns).intersection(t.columns)
    )

    print("Common columns:", len(common))

    print("\nAnalysis-only columns:")
    print(sorted(set(a.columns) - set(t.columns)))

    print("\nTrace-only columns:")
    print(sorted(set(t.columns) - set(a.columns)))

    print("\nCommon columns:")
    print(common)

DXT SOURCE COMPARISON

Analysis files:

Trace files:
